# 📡 TelecomX — Parte 2: Modelos Predictivos de Churn

> **Objetivo:** Construir, evaluar e interpretar modelos de Machine Learning para predecir el abandono (churn) de clientes de TelecomX, a partir del dataset tratado en la Parte 1.

---

| Sección | Contenido |
|---|---|
| 🛠️ Preparación | Carga, limpieza, encoding y balanceo |
| 🎯 Correlación | Análisis de variables relevantes |
| 🤖 Modelado | Regresión Logística, Random Forest, Árbol de Decisión |
| 📋 Interpretación | Importancia de variables y conclusiones |

## ⚙️ Configuración Inicial

In [ ]:
# === Librerías ===
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score,
    ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE

# Visualización
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

print('✅ Librerías cargadas correctamente.')

---
# 🛠️ SECCIÓN 1 — PREPARACIÓN DE LOS DATOS

## 1.1 — Carga del archivo tratado

In [ ]:
# Cargar datos tratados de la Parte 1
df = pd.read_csv('datos_tratados.csv')

print(f'✅ Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head(3)

In [ ]:
# Verificación general
print('📋 Tipos de dato:')
print(df.dtypes)
print(f'\n🔍 Valores nulos:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

## 1.2 — Estandarización de nombres de columnas

In [ ]:
# Renombrar columnas anidadas al formato plano
df.columns = [
    col.replace('customer.', '')
       .replace('phone.', '')
       .replace('internet.', '')
       .replace('account.', '')
       .replace('Charges.', 'Charges_')
    for col in df.columns
]

print('✅ Columnas renombradas:')
print(df.columns.tolist())

## 1.3 — Eliminación de Columnas Irrelevantes

In [ ]:
# customerID no aporta información predictiva
df.drop(columns=['customerID'], inplace=True)

# Tratar nulos residuales en Charges_Total
df['Charges_Total'] = pd.to_numeric(df['Charges_Total'], errors='coerce')
df['Charges_Monthly'] = pd.to_numeric(df['Charges_Monthly'], errors='coerce')
df['Charges_Total'].fillna(df['Charges_Monthly'], inplace=True)
df.dropna(inplace=True)

print(f'✅ Dataset limpio: {df.shape[0]} registros × {df.shape[1]} columnas')

## 1.4 — Encoding de Variables Categóricas

In [ ]:
# Separar variable objetivo
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Identificar columnas categóricas (excluyendo target)
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'🔠 Columnas categóricas a encodear ({len(cat_cols)}): {cat_cols}')

# Label Encoding para binarias / One-Hot para multi-clase
binary_cols   = [c for c in cat_cols if df[c].nunique() == 2]
multicat_cols = [c for c in cat_cols if df[c].nunique() > 2]

le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

df = pd.get_dummies(df, columns=multicat_cols, drop_first=True)

# Convertir booleanos a int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f'\n✅ Encoding completo. Nuevas dimensiones: {df.shape}')
print('Columnas finales:')
print(df.columns.tolist())

## 1.5 — Verificación de la Proporción de Churn

In [ ]:
churn_dist = df['Churn'].value_counts()
churn_pct  = df['Churn'].value_counts(normalize=True) * 100

print('📊 Distribución de Churn:')
for k, label in [(0, 'No Churn'), (1, 'Churn')]:
    print(f'  {label}: {churn_dist[k]:>5} clientes ({churn_pct[k]:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 5))
colors = ['#2ecc71', '#e74c3c']
bars = ax.bar(['No Churn (0)', 'Churn (1)'], churn_dist.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, val, pct in zip(bars, churn_dist.values, churn_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
            f'{val}\n({pct:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Proporción de Churn en el Dataset')
ax.set_ylabel('Cantidad de Clientes')
plt.tight_layout()
plt.show()

## 1.6 — Balanceo de Clases con SMOTE

> El dataset presenta desbalance (~26% Churn vs ~74% No Churn). Aplicamos **SMOTE** para sobremuestrear la clase minoritaria en el conjunto de entrenamiento.

In [ ]:
# Separar features y target ANTES del balanceo
X = df.drop(columns=['Churn'])
y = df['Churn']

# División train/test (80/20 — estratificada)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Aplicar SMOTE solo sobre train
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('📦 Antes de SMOTE (train):', dict(pd.Series(y_train).value_counts()))
print('📦 Después de SMOTE (train):', dict(pd.Series(y_train_bal).value_counts()))
print(f'\n✅ Train balanceado: {X_train_bal.shape[0]} muestras')
print(f'   Test (sin tocar):  {X_test.shape[0]} muestras')

## 1.7 — Escalado de Variables Numéricas

In [ ]:
scaler = StandardScaler()
num_cols = ['tenure', 'Charges_Monthly', 'Charges_Total']

X_train_bal[num_cols] = scaler.fit_transform(X_train_bal[num_cols])
X_test[num_cols]      = scaler.transform(X_test[num_cols])

print('✅ Escalado aplicado a:', num_cols)

---
# 🎯 SECCIÓN 2 — CORRELACIÓN Y SELECCIÓN DE VARIABLES

## 2.1 — Mapa de Correlación General

In [ ]:
# Correlación de todas las variables con Churn (ordenado)
corr_churn = df.corr()['Churn'].drop('Churn').sort_values(key=abs, ascending=False)

print('🔗 Top 15 variables más correlacionadas con Churn:')
print(corr_churn.head(15).to_string())

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in corr_churn.head(15).values]
ax.barh(corr_churn.head(15).index[::-1],
        corr_churn.head(15).values[::-1],
        color=colors[::-1], edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 15 Variables — Correlación con Churn')
ax.set_xlabel('Correlación de Pearson')
plt.tight_layout()
plt.show()

## 2.2 — Heatmap de Correlación (variables numéricas clave)

In [ ]:
# Seleccionar top variables para heatmap
top_vars = corr_churn.head(12).index.tolist() + ['Churn']
corr_matrix = df[top_vars].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, center=0, linewidths=0.5, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Heatmap de Correlación — Variables Seleccionadas', fontsize=14)
plt.tight_layout()
plt.show()

## 2.3 — Análisis Dirigido: Variables Clave vs. Churn

In [ ]:
# Distribución de tenure y Charges_Monthly por clase Churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
    subset = df[df['Churn'] == label]
    axes[0].hist(subset['tenure'], bins=30, alpha=0.6, color=color,
                 label=f'Churn={label}')
    axes[1].hist(subset['Charges_Monthly'], bins=30, alpha=0.6, color=color,
                 label=f'Churn={label}')

axes[0].set_title('Distribución de Tenure por Churn')
axes[0].set_xlabel('Meses de contrato (tenure)')
axes[0].legend()

axes[1].set_title('Distribución de Cargo Mensual por Churn')
axes[1].set_xlabel('Charges Monthly (USD)')
axes[1].legend()

plt.suptitle('Análisis Dirigido — Variables Numéricas vs. Churn', fontweight='bold')
plt.tight_layout()
plt.show()

---
# 🤖 SECCIÓN 3 — MODELADO PREDICTIVO

## 3.1 — Definición de Modelos

Se entrenarán y compararán tres modelos:

| Modelo | Tipo | Ventaja Principal |
|---|---|---|
| Regresión Logística | Lineal | Interpretable, rápido |
| Árbol de Decisión | No lineal | Explícito, visual |
| Random Forest | Ensemble | Alta precisión, robusto |

In [ ]:
# Definir modelos
modelos = {
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
    'Árbol de Decisión':   DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, max_depth=10,
                                                   random_state=42, n_jobs=-1)
}

resultados = {}

for nombre, modelo in modelos.items():
    modelo.fit(X_train_bal, y_train_bal)
    y_pred  = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    resultados[nombre] = {
        'modelo': modelo,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'accuracy': acc,
        'auc': auc
    }
    print(f'✅ {nombre:25} | Accuracy: {acc:.4f} | AUC-ROC: {auc:.4f}')

## 3.2 — Evaluación: Reportes de Clasificación

In [ ]:
for nombre, res in resultados.items():
    print(f'\n{'='*55}')
    print(f'📊 {nombre}')
    print('='*55)
    print(classification_report(y_test, res['y_pred'],
                                  target_names=['No Churn', 'Churn']))

## 3.3 — Matrices de Confusión

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (nombre, res) in zip(axes, resultados.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['No Churn', 'Churn'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{nombre}\nACC={res["accuracy"]:.3f} | AUC={res["auc"]:.3f}')

plt.suptitle('Matrices de Confusión — Comparación de Modelos',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.4 — Curvas ROC Comparativas

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
colors_roc = ['#3498db', '#e67e22', '#27ae60']

for (nombre, res), color in zip(resultados.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{nombre} (AUC = {res['auc']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Clasificador Aleatorio')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlabel('Tasa de Falsos Positivos (FPR)')
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR)')
ax.set_title('Curvas ROC — Comparación de Modelos', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 3.5 — Validación Cruzada (Cross-Validation)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('📊 Validación Cruzada (5-fold) — AUC-ROC:')
print('-'*50)

cv_results = {}
for nombre, res in resultados.items():
    scores = cross_val_score(res['modelo'], X, y, cv=cv,
                              scoring='roc_auc', n_jobs=-1)
    cv_results[nombre] = scores
    print(f'  {nombre:25} | Media: {scores.mean():.4f} ± {scores.std():.4f}')

# Boxplot comparativo
fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(cv_results.values(), labels=cv_results.keys(), patch_artist=True,
           boxprops=dict(facecolor='#3498db', alpha=0.6),
           medianprops=dict(color='#e74c3c', linewidth=2))
ax.set_title('Distribución AUC-ROC — Cross Validation (5-fold)')
ax.set_ylabel('AUC-ROC')
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

## 3.6 — Tabla Resumen de Métricas

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

resumen_metricas = []
for nombre, res in resultados.items():
    resumen_metricas.append({
        'Modelo': nombre,
        'Accuracy': round(res['accuracy'], 4),
        'Precision (Churn)': round(precision_score(y_test, res['y_pred']), 4),
        'Recall (Churn)':    round(recall_score(y_test, res['y_pred']), 4),
        'F1-Score (Churn)':  round(f1_score(y_test, res['y_pred']), 4),
        'AUC-ROC':           round(res['auc'], 4),
        'CV AUC (media)':    round(cv_results[nombre].mean(), 4)
    })

df_metricas = pd.DataFrame(resumen_metricas).set_index('Modelo')
print('📋 Tabla Comparativa de Métricas:')
display(df_metricas.style.highlight_max(color='#d4efdf').highlight_min(color='#fadbd8'))

---
# 📋 SECCIÓN 4 — INTERPRETACIÓN Y CONCLUSIONES

> ⏳ **Esta sección se completará después de revisar los resultados obtenidos en los modelos.**

## 4.1 — Importancia de Variables (Random Forest)

In [ ]:
# Feature importance del mejor modelo esperado (Random Forest)
rf_model = resultados['Random Forest']['modelo']
importancias = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print('🏆 Top 15 variables más importantes (Random Forest):')
print(importancias.head(15).to_string())

fig, ax = plt.subplots(figsize=(10, 7))
importancias.head(15)[::-1].plot(kind='barh', ax=ax,
                                   color=sns.color_palette('viridis', 15))
ax.set_title('Importancia de Variables — Random Forest (Top 15)')
ax.set_xlabel('Importancia (Gini)')
plt.tight_layout()
plt.show()

## 4.2 — Coeficientes de Regresión Logística

In [ ]:
# Coeficientes de Regresión Logística (interpretabilidad)
rl_model = resultados['Regresión Logística']['modelo']
coeficientes = pd.Series(
    rl_model.coef_[0],
    index=X.columns
).sort_values(key=abs, ascending=False)

print('📐 Top 15 coeficientes — Regresión Logística:')
print(coeficientes.head(15).to_string())

fig, ax = plt.subplots(figsize=(10, 7))
top_coef = coeficientes.head(15)[::-1]
colors_coef = ['#e74c3c' if v > 0 else '#3498db' for v in top_coef.values]
ax.barh(top_coef.index, top_coef.values, color=colors_coef, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Coeficientes — Regresión Logística (Top 15)\n'
             '🔴 Aumentan Churn | 🔵 Reducen Churn')
ax.set_xlabel('Coeficiente')
plt.tight_layout()
plt.show()

## 4.3 — Conclusiones

> 📝 **Pendiente:** Esta sección se completará con los resultados reales de los modelos.

---

### 🏆 Mejor Modelo
*(Se completará con los outputs reales)*

### 🔎 Variables Más Influyentes
*(Se completará con los outputs reales)*

### 💡 Recomendaciones de Negocio
*(Se completará con los outputs reales)*